코드 11
ViT(Vision Transformer) 단독 성능 측정

** Best 퍼포먼스 **

Epoch [40/50] Train Loss: 1.1955 Train Acc: 0.9324 | Val Loss: 4.0975 Val Acc: 0.8226
=== 최종 평가 결과 ===
Base Validation Acc : 0.8226 (82.26% 와 동일해야 함)
TTA Validation Acc  : 0.8235


In [1]:
import os
import math
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm

# 디바이스 설정 (서버 GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
# # ViT 전용 224x224 전처리 파이프라인
# train_transform = A.Compose([
#     A.Resize(224, 224),
#     A.HorizontalFlip(p=0.5),
#     A.RandomBrightnessContrast(p=0.2),
#     A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#     ToTensorV2()
# ])

# Cell 2: train_transform 수정
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5), # 색상 변형 추가
    A.CoarseDropout(max_holes=2, max_height=32, max_width=32, fill_value=0, p=0.5), # 이미지 일부를 검은 박스로 가림
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class EmotionDataset(Dataset):
    def __init__(self, img_paths, labels, transform=None):
        self.img_paths = img_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
            
        label = self.labels[idx]
        return image, torch.tensor(label, dtype=torch.long)

/tmp/ipykernel_2972365/1177159750.py:15: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=2, max_height=32, max_width=32, fill_value=0, p=0.5), # 이미지 일부를 검은 박스로 가림


In [3]:
import os
import glob

# 1. v8_arcface 학습 시 사용했던 최신 크롭 데이터셋 베이스 경로
base_dir = '/workspace/user1/dataset_v3_cropped'
train_dir = os.path.join(base_dir, 'train')
valid_dir = os.path.join(base_dir, 'val')

# 2. 클래스 매핑 (폴더명의 대소문자 차이로 인한 누락을 방지하기 위해 소문자 기준 매핑)
emotion_map = {'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3}

def get_data_list(root_dir):
    paths = []
    labels = []
    
    if not os.path.exists(root_dir):
        print(f"⚠️ 경로를 찾을 수 없습니다: {root_dir}")
        return paths, labels
        
    for folder_name in os.listdir(root_dir):
        emotion_key = folder_name.lower()
        if emotion_key in emotion_map:
            label_idx = emotion_map[emotion_key]
            # 해당 감정 폴더 하위의 모든 이미지 수집
            folder_path = os.path.join(root_dir, folder_name, '*')
            img_files = glob.glob(folder_path)
            for f in img_files:
                paths.append(f)
                labels.append(label_idx)
                
    return paths, labels

# 3. 데이터 로딩 (train/val 분리된 폴더에서 각각 긁어옴)
train_img_paths, train_labels = get_data_list(train_dir)
valid_img_paths, valid_labels = get_data_list(valid_dir)

print(f"데이터 로드 완료! Train: {len(train_img_paths)}장, Valid: {len(valid_img_paths)}장")

# 4. Dataset 및 DataLoader 생성 (ViT 학습용 224x224 파이프라인 적용)
train_dataset = EmotionDataset(img_paths=train_img_paths, labels=train_labels, transform=train_transform)
valid_dataset = EmotionDataset(img_paths=valid_img_paths, labels=valid_labels, transform=valid_transform)

batch_size = 32

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

데이터 로드 완료! Train: 5329장, Valid: 1037장


In [4]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50, easy_margin=False):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.easy_margin = easy_margin
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        one_hot = torch.zeros(cosine.size(), device=device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        
        # Loss 계산용 output과 Acc 계산용 순수 cosine을 같이 반환
        return output, cosine

class ViTArcFace(nn.Module):
    def __init__(self, num_classes=4, m=0.7):
        super(ViTArcFace, self).__init__()
        # self.backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        # drop_path_rate=0.2 추가
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0, drop_path_rate=0.2)
        embed_dim = self.backbone.num_features 
        
        self.arc_face_head = ArcMarginProduct(in_features=embed_dim, out_features=num_classes, s=30.0, m=m)

    def forward(self, x, labels=None):
        features = self.backbone(x)
        if labels is not None:
            # 학습/검증 시: Loss용 output과 Acc용 순수 각도 동시 획득
            output, cosine = self.arc_face_head(features, labels)
            return output, cosine
        
        # 실전 추론 시: 순수 각도 반환 (소프트맥스용)
        cosine = F.linear(F.normalize(features), F.normalize(self.arc_face_head.weight))
        return cosine * self.arc_face_head.s

In [5]:
# 기존 모델 초기화 코드는 그대로 유지
model = ViTArcFace(num_classes=4, m=0.5).to(device)

# Loss 함수 (ArcFace에 집중하기 위해 label_smoothing 제거)
criterion = nn.CrossEntropyLoss()

# --- 가중치 동결 로직 완화 ---
for name, param in model.backbone.named_parameters():
    # 패치 임베딩 등 기초 변환 층은 계속 동결 유지
    if 'patch_embed' in name or 'cls_token' in name or 'pos_embed' in name:
        param.requires_grad = False
    
    # 🔴 수정 포인트: 12개의 블록 중 앞부분 0~5번 블록만 동결 (뒤쪽 6~11번 블록은 학습 허용)
    if 'blocks' in name:
        block_idx = int(name.split('.')[1])
        if block_idx < 6: # 9에서 6으로 완화
            param.requires_grad = False

trainable_params = filter(lambda p: p.requires_grad, model.parameters())

# Optimizer 선언 (학습률은 3e-5 유지)
optimizer = torch.optim.AdamW(trainable_params, lr=3e-5, weight_decay=5e-2)

# --- Scheduler 및 에포크 수정 ---
# 에포크를 50으로 늘리므로 스케줄러의 T_0 값도 조금 늘려줍니다.
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=15, T_mult=2, eta_min=1e-6
)

epochs = 50

scaler = torch.cuda.amp.GradScaler() # AMP 연산을 위한 스케일러 객체 생성

/tmp/ipykernel_2972365/3264600999.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # AMP 연산을 위한 스케일러 객체 생성


In [6]:
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # 🔴 AMP 적용: 연산량이 많은 부분만 16비트로 자동 전환
        with torch.cuda.amp.autocast():
            outputs, cosine = model(images, labels) 
            loss = criterion(outputs, labels)
        
        # 🔴 역전파도 scaler를 통해서 진행
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        _, predicted = cosine.max(1) 
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    train_acc = correct / total
    
    # ... (이하 validation 루프는 동일, 검증 루프에도 autocast를 씌우면 더 빠름)
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]"):
            images, labels = images.to(device), labels.to(device)
            
            with torch.cuda.amp.autocast(): # 검증 루프에도 추가
                outputs, cosine = model(images, labels)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = cosine.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    val_acc = correct / total
    scheduler.step()
    
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {train_loss/len(train_loader):.4f} Train Acc: {train_acc:.4f} | Val Loss: {val_loss/len(valid_loader):.4f} Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'vit_arcface_best.pth')
        print(f"★ Best Model Saved! (Acc: {val_acc:.4f})")

Epoch 1/50 [Train]:   0%|          | 0/167 [00:00<?, ?it/s]/tmp/ipykernel_2972365/4248620232.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/50 [Valid]:   0%|          | 0/33 [00:00<?, ?it/s]/tmp/ipykernel_2972365/4248620232.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): # 검증 루프에도 추가
Epoch 1/50 [Valid]: 100%|██████████| 33/33 [00:03<00:00,  9.10it/s]


Epoch [1/50] Train Loss: 11.5951 Train Acc: 0.5177 | Val Loss: 7.7730 Val Acc: 0.6895
★ Best Model Saved! (Acc: 0.6895)


Epoch 2/50 [Valid]: 100%|██████████| 33/33 [00:03<00:00,  9.77it/s]


Epoch [2/50] Train Loss: 7.8339 Train Acc: 0.6816 | Val Loss: 6.1914 Val Acc: 0.7387
★ Best Model Saved! (Acc: 0.7387)


Epoch 3/50 [Valid]: 100%|██████████| 33/33 [00:02<00:00, 11.35it/s]


Epoch [3/50] Train Loss: 6.4391 Train Acc: 0.7459 | Val Loss: 5.2503 Val Acc: 0.7869
★ Best Model Saved! (Acc: 0.7869)


... (90 outputs left out)

Epoch 49/50 [Valid]: 100%|██████████| 33/33 [00:04<00:00,  7.05it/s]


Epoch [49/50] Train Loss: 1.6439 Train Acc: 0.9067 | Val Loss: 4.0022 Val Acc: 0.8120


Epoch 50/50 [Valid]: 100%|██████████| 33/33 [00:03<00:00,  8.29it/s]

Epoch [50/50] Train Loss: 1.6251 Train Acc: 0.9017 | Val Loss: 3.8498 Val Acc: 0.8062


In [7]:
# TTA 적용

import torch

# 1. 최고 성능 가중치 로드
model.load_state_dict(torch.load('vit_arcface_best.pth'))
model.eval()

correct_base = 0
correct_tta = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(valid_loader, desc="[Evaluating with TTA]"):
        images, labels = images.to(device), labels.to(device)
        
        # --- (A) 원본 이미지 추론 ---
        with torch.cuda.amp.autocast():
            _, cosine_orig = model(images, labels)
        
        # 단일 모델(Base) 예측 기록
        _, predicted_base = cosine_orig.max(1)
        correct_base += predicted_base.eq(labels).sum().item()

        # --- (B) 좌우 반전 이미지 추론 ---
        # Tensor의 4번째 차원(Width)을 반전시킵니다 [Batch, Channel, Height, Width]
        images_flipped = torch.flip(images, dims=[3]) 
        
        with torch.cuda.amp.autocast():
            _, cosine_flipped = model(images_flipped, labels)
            
        # --- (C) TTA 앙상블 (점수 평균화) ---
        # 두 각도(유사도) 점수를 더해서 평균을 냅니다.
        cosine_tta = (cosine_orig + cosine_flipped) / 2.0
        
        # TTA 적용 후 예측 기록
        _, predicted_tta = cosine_tta.max(1)
        correct_tta += predicted_tta.eq(labels).sum().item()
        
        total += labels.size(0)

base_acc = correct_base / total
tta_acc = correct_tta / total

print("\n=== 최종 평가 결과 ===")
print(f"Base Validation Acc : {base_acc:.4f} (82.26% 와 동일해야 함)")
print(f"TTA Validation Acc  : {tta_acc:.4f}")

[Evaluating with TTA]:   0%|          | 0/33 [00:00<?, ?it/s]/tmp/ipykernel_2972365/851518155.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_2972365/851518155.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[Evaluating with TTA]: 100%|██████████| 33/33 [00:03<00:00, 10.69it/s]


=== 최종 평가 결과 ===
Base Validation Acc : 0.8226 (82.26% 와 동일해야 함)
TTA Validation Acc  : 0.8235
